# Deployment di un Prompt Agent su Microsoft Foundry

Crea una versione di un **Prompt Agent minimale**, con solo modello e istruzioni, nel progetto distribuito da [infra/main.bicep](infra/main.bicep). Non configura tool, connessioni MCP o accesso a knowledge base e non crea altre risorse Foundry o modelli.

## Prerequisiti

1. Completa il deployment descritto in [infra/README.md](infra/README.md).
2. Nel kernel selezionato installa le dipendenze: `%pip install "azure-ai-projects>=2.0.0" azure-identity`. Riavvia il kernel dopo un aggiornamento dei pacchetti gia' importati. Serve Python 3.10 o successivo.
3. Esegui `az login --tenant <tenant-id>` e seleziona la sottoscrizione corretta. Il notebook usa **AzureCliCredential**, non chiavi.
4. L'utente deve avere **Foundry User** per creare l'agente. Se abiliti la lettura degli output ARM, servono anche permessi di lettura sul deployment.

Le istruzioni predefinite sono mantenute per il successivo collegamento a una knowledge base. Da sole non forniscono accesso alle fonti: il collegamento deve essere configurato separatamente prima di usare l'agente per risposte basate sui documenti.

**Effetti:** ogni esecuzione della cella di deployment crea una nuova versione dell'agente. Non elimina risorse e non esegue inferenza o test di conversazione.


## 1. Configurazione

Nomi delle risorse e istruzioni sono variabili nelle due celle seguenti. Verifica i valori precompilati. Con `LOAD_BICEP_OUTPUTS=True`, il notebook legge gli output del deployment ARM e completa i valori vuoti di tenant, account, progetto e modello; i valori inseriti esplicitamente hanno precedenza. `BICEP_DEPLOYMENT_NAME` identifica il deployment ARM da leggere, non l'agente.

Per usare solo la configurazione manuale, imposta `LOAD_BICEP_OUTPUTS=False` e compila tutti i valori. I default del Bicep sono progetto `wpc-foundry-iq-demo` e deployment chat `gpt-5-mini`; il nome dell'account Foundry contiene un suffisso generato.


In [ ]:
SUBSCRIPTION_ID = ""
RESOURCE_GROUP_NAME = "20261124-wpc-foundryiq"
BICEP_DEPLOYMENT_NAME = "wpc-foundry-iq-demo"
LOAD_BICEP_OUTPUTS = True

TENANT_ID = ""
FOUNDRY_ACCOUNT_NAME = ""
FOUNDRY_PROJECT_NAME = "wpc-foundry-iq-demo"
MODEL_DEPLOYMENT_NAME = "gpt-5-mini"

AGENT_NAME = "wpc-technician-assistant"


In [ ]:
AGENT_INSTRUCTIONS = """You are a field service assistant for industrial machines and spare parts.

Always query and use only the knowledge base tool to answer (send only english prompts) — do not rely on your own knowledge or any external sources.

Cite every source inline using this exact format: [message_idx:source_idx|source_name]. Include all sources you used.

If the knowledge base does not contain the answer, say "I don't know." (in the user's language).

Answer in the user's language, almost 120 words"""


## 2. Risoluzione degli output e autenticazione

Se `LOAD_BICEP_OUTPUTS=True`, questa cella legge gli output ARM senza ridistribuire il Bicep. Altrimenti usa solo i valori configurati. Inizializza poi il client del progetto. Il tenant deve corrispondere a quello delle risorse. In caso di `403`, controlla identita', scope dei ruoli e propagazione RBAC.


In [ ]:
import json
import shutil
import subprocess
from urllib.parse import quote
from uuid import UUID

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential

UUID(SUBSCRIPTION_ID)

if LOAD_BICEP_OUTPUTS:
    az_executable = shutil.which("az")
    if not az_executable:
        raise RuntimeError("Azure CLI non trovata. Installala oppure configura manualmente i valori del notebook.")
    result = subprocess.run(
        [az_executable, "deployment", "group", "show",
         "--subscription", SUBSCRIPTION_ID,
         "--resource-group", RESOURCE_GROUP_NAME,
         "--name", BICEP_DEPLOYMENT_NAME,
         "--query", "properties.outputs", "--output", "json", "--only-show-errors"],
        capture_output=True, text=True, check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Lettura degli output Bicep fallita: {result.stderr.strip()}")
    bicep_outputs = json.loads(result.stdout)
    demo_config = bicep_outputs["demoConfig"]["value"]
    TENANT_ID = TENANT_ID or demo_config["FOUNDRYIQ_DEMO_TENANT_ID"]
    FOUNDRY_ACCOUNT_NAME = FOUNDRY_ACCOUNT_NAME or bicep_outputs["foundryAccountName"]["value"]
    FOUNDRY_PROJECT_NAME = FOUNDRY_PROJECT_NAME or bicep_outputs["foundryProjectName"]["value"]
    MODEL_DEPLOYMENT_NAME = MODEL_DEPLOYMENT_NAME or demo_config["FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT"]

required_names = {
    "RESOURCE_GROUP_NAME": RESOURCE_GROUP_NAME,
    "FOUNDRY_ACCOUNT_NAME": FOUNDRY_ACCOUNT_NAME,
    "FOUNDRY_PROJECT_NAME": FOUNDRY_PROJECT_NAME,
    "MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
    "AGENT_NAME": AGENT_NAME,
}
for variable, value in required_names.items():
    if not value or value != value.strip() or any(char in value for char in "/\\?#<>"):
        raise ValueError(f"Imposta un nome valido per {variable} nella cella di configurazione.")
UUID(TENANT_ID)
if not AGENT_INSTRUCTIONS.strip():
    raise ValueError("AGENT_INSTRUCTIONS non puo' essere vuoto.")

PROJECT_ENDPOINT = (
    f"https://{FOUNDRY_ACCOUNT_NAME}.services.ai.azure.com/api/projects/{quote(FOUNDRY_PROJECT_NAME, safe='')}"
)

credential = AzureCliCredential(tenant_id=TENANT_ID)
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
print(json.dumps({
    "project_endpoint": PROJECT_ENDPOINT,
    "model_deployment": MODEL_DEPLOYMENT_NAME,
    "agent": AGENT_NAME,
}, indent=2))


## 3. Deployment del Prompt Agent

Ogni esecuzione crea una nuova versione, anche se il nome dell'agente esiste gia'. Non viene creato un container o un endpoint applicativo separato. Gli altri client possono invocare la versione stampata con la Responses API del progetto.


In [ ]:
print(PROJECT_ENDPOINT)
print([a.name for a in project_client.agents.list()])

In [ ]:
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True)

definition = PromptAgentDefinition(
    model=MODEL_DEPLOYMENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
)

agent = project_client.agents.create_version(agent_name=AGENT_NAME, definition=definition)
AGENT_VERSION = agent.version
print(f"Prompt Agent distribuito: {agent.name}, versione {AGENT_VERSION}")
print(f"Progetto: {PROJECT_ENDPOINT}")


In [ ]:
# Test a simple chat with the agent

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = PROJECT_ENDPOINT
agent_name = AGENT_NAME
user_message = f"Hi {AGENT_NAME}"

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(
        endpoint=endpoint,
        credential=credential,
        allow_preview=True,
    ) as project_client,
):
    # Bind the OpenAI client to the existing agent endpoint.
    openai_client = project_client.get_openai_client(agent_name=agent_name)

    # GitHub Copilot preview agents require a conversation for every Responses turn.
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=[{"role": "user", "content": user_message}],
    )

    print(f"Response output: {response.output_text}")

## Riferimenti e chiusura

- [Infrastruttura e ruoli della demo](infra/README.md)

Il notebook lascia la versione dell'agente disponibile in Foundry, senza tool o connessioni aggiunti. La cella seguente chiude solo i client locali, senza cancellare risorse.


In [ ]:
project_client.close()
credential.close()
